# ETL — Leitos Hospitalares

**Diferença em relação ao ETL:**
- No **ETL**, os dados são transformados em Python antes de entrar no banco.
- No **ELT**, os dados brutos entram no banco primeiro (schema `raw`), e todas as transformações acontecem dentro do PostgreSQL via **Views SQL** (schema `elt`).

**Fluxo:**
```
CSVs → Python (Pandas: Limpeza, Sets, Joins e Modelagem) → PostgreSQL (etl.dim_* + etl.fato_*)
```

## 1. Instalação e imports

In [2]:
%pip install pandas sqlalchemy psycopg2-binary python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd #principal biblioteca para manipulação dos CSVs e dados
import os
import csv 
from dotenv import load_dotenv #busca as variaveis do arquivo .env
from sqlalchemy import create_engine, text #conecta Python ao banco  

## 2 EXTRACT — extração e união dos dados

### 2.1 Funções auxiliares

Criando dicionario com os paths para os CSVs para ser usado futuramente

In [4]:
ARQUIVOS_LEITOS = {
    2023: "../database/data_raw/Leitos_2023.csv",
    2024: "../database/data_raw/Leitos_2024.csv",
    2025: "../database/data_raw/Leitos_2025.csv"
} 

Função para detectar automaticamente o separador dos arquivos de Anos diferentes, evitando erro na hora de concatenar

2023 e 2024 usam '**,**' | 2025 usa '**;**'

In [5]:
def detectar_separador(caminho_arquivo):
    with open(caminho_arquivo, "r", encoding="latin-1", newline="") as arquivo:
        amostra = arquivo.read(4096)

    separador = csv.Sniffer().sniff(amostra, delimiters=",;").delimiter

    return separador

Função que padroniza os nomes das colunas para facilitar consultas SQL

Exemplo:                    
    NOME_ESTABELECIMENTO → nome_estabelecimento                     
    LEITOS_EXISTENTES → leitos_existentes                   


In [6]:
def padronizar_nome_colunas(df):
    
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
    )

    return df

In [7]:
def ler_csv_leitos(caminho_arquivo):
    """
    Lê um arquivo CSV de leitos.

    Essa função:
    1. Detecta o separador do arquivo
    2. Lê o CSV com encoding latin-1
    3. Mantém códigos como texto
    4. Padroniza nomes das colunas
    """

    separador = detectar_separador(caminho_arquivo)

    df = pd.read_csv(
        caminho_arquivo,
        sep=separador,
        encoding="latin-1",
        low_memory=False
    )

    df = padronizar_nome_colunas(df)

    return df

4.3 função para os três arquivos

In [8]:
bases = []

for caminho in ARQUIVOS_LEITOS.items():
    df_ano = ler_csv_leitos(caminho[1])
    bases.append(df_ano)

print("DataFrames lidos:")
for i, df in enumerate(bases):
    print(f"  - DataFrame {i+2023}: {df.shape[0]} linhas e {df.shape[1]} colunas")

DataFrames lidos:
  - DataFrame 2023: 84471 linhas e 34 colunas
  - DataFrame 2024: 85225 linhas e 34 colunas
  - DataFrame 2025: 86147 linhas e 35 colunas


In [9]:
# Comparando as colunas entre os DataFrames
dif_23_24 = set(bases[0].columns) - set(bases[1].columns)
print("Colunas que faltam em 2024 em relação a 2023:", dif_23_24)
new_23_24 = set(bases[1].columns) - set(bases[0].columns)
print("Colunas novas em 2024 em relação a 2023:", new_23_24)
print("-" * 50)

dif_23_25 = set(bases[0].columns) - set(bases[2].columns)
print("Colunas que faltam em 2025 em relação a 2023:", dif_23_25)
new_23_25 = set(bases[2].columns) - set(bases[0].columns)
print("Colunas novas em 2025 em relação a 2023:", new_23_25)
print("-" * 50)

dif_24_25 = set(bases[1].columns) - set(bases[2].columns)
print("Colunas que faltam em 2025 em relação a 2024:", dif_24_25)
new_24_25 = set(bases[2].columns) - set(bases[1].columns)
print("Colunas novas em 2025 em relação a 2024:", new_24_25)

Colunas que faltam em 2024 em relação a 2023: set()
Colunas novas em 2024 em relação a 2023: set()
--------------------------------------------------
Colunas que faltam em 2025 em relação a 2023: set()
Colunas novas em 2025 em relação a 2023: {'co_ibge'}
--------------------------------------------------
Colunas que faltam em 2025 em relação a 2024: set()
Colunas novas em 2025 em relação a 2024: {'co_ibge'}


In [10]:
# Removendo colunas que não existem em todas as bases
bases[2].drop("co_ibge", axis=1, inplace=True)

# Juntando tudo em um único DataFrame
df_raw = pd.concat(bases, ignore_index=True, sort=False)

bases → lista que guarda os DataFrames de 2023, 2024 e 2025           
pd.concat → empilha todos em uma base única           
ignore_index=True → recria o índice                 
sort=False → não reorganiza colunas automaticamente                       

4.4 Verifiçao

In [11]:
print(f"DataFrame unificado: {df_raw.shape[0]} linhas e {df_raw.shape[1]} colunas")
print("-" * 50)
df_raw.head()

DataFrame unificado: 255843 linhas e 34 colunas
--------------------------------------------------


,comp,regiao,uf,municipio,motivo_desabilitacao,cnes,nome_estabelecimento,razao_social,tp_gestao,co_tipo_unidade,...,uti_adulto_exist,uti_adulto_sus,uti_pediatrico_exist,uti_pediatrico_sus,uti_neonatal_exist,uti_neonatal_sus,uti_queimado_exist,uti_queimado_sus,uti_coronariana_exist,uti_coronariana_sus
0,202301,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,27,CASA DE SAUDE SANTA HELENA,CASA DE SAUDE E MATERNIDADE SANTA HELENA LTDA,M,5,...,0,0,0,0,0,0,0,0,0,0
1,202301,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,35,HOSPITAL MENDO SAMPAIO,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,M,5,...,0,0,0,0,0,0,0,0,0,0
2,202301,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,94,MATERNIDADE PADRE GERALDO LEITE BASTOS,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,M,7,...,0,0,0,0,0,0,0,0,0,0
3,202301,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,183,HOSPITAL SAMARITANO,SOCIEDADE HOSPITALAR SAMARITANO LTDA,M,5,...,5,0,0,0,0,0,0,0,0,0
4,202301,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,221,HOSPITAL SAO SEBASTIAO,CASA DE SAUDE E MATERNIDADE SAO SEBASTIAO LTDA,M,5,...,10,0,0,0,0,0,0,0,0,0


5. Diagnóstico inicial dos dados

Análise da base:

In [12]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 255843 entries, 0 to 255842
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   comp                    255843 non-null  int64  
 1   regiao                  255843 non-null  str    
 2   uf                      255843 non-null  str    
 3   municipio               255843 non-null  str    
 4   motivo_desabilitacao    0 non-null       float64
 5   cnes                    255843 non-null  int64  
 6   nome_estabelecimento    255841 non-null  str    
 7   razao_social            255843 non-null  str    
 8   tp_gestao               255843 non-null  str    
 9   co_tipo_unidade         255843 non-null  int64  
 10  ds_tipo_unidade         255843 non-null  str    
 11  natureza_juridica       255843 non-null  int64  
 12  desc_natureza_juridica  255843 non-null  str    
 13  no_logradouro           255843 non-null  str    
 14  nu_endereco             255843 

In [13]:
print(df_raw.dtypes)

comp                        int64
regiao                        str
uf                            str
municipio                     str
motivo_desabilitacao      float64
cnes                        int64
nome_estabelecimento          str
razao_social                  str
tp_gestao                     str
co_tipo_unidade             int64
ds_tipo_unidade               str
natureza_juridica           int64
desc_natureza_juridica        str
no_logradouro                 str
nu_endereco                   str
no_complemento                str
no_bairro                     str
co_cep                      int64
nu_telefone                   str
no_email                      str
leitos_existentes           int64
leitos_sus                  int64
uti_total_exist             int64
uti_total_sus               int64
uti_adulto_exist            int64
uti_adulto_sus              int64
uti_pediatrico_exist        int64
uti_pediatrico_sus          int64
uti_neonatal_exist          int64
uti_neonatal_s

Serve para ver:                         

quantidade de linhas                
nomes das colunas           
tipos das colunas                            
quantidade de nulos                                             

6.1 Ver nulos

In [14]:
# Cria a tabela de diagnóstico de nulos
tabela_nulos = pd.DataFrame({
    'Total Nulos': df_raw.isna().sum(),
    'Porcentagem %': ((df_raw.isna().sum() / len(df_raw)) * 100).round(2)
}).sort_values(by='Total Nulos', ascending=False)

print(tabela_nulos)

                        Total Nulos  Porcentagem %
motivo_desabilitacao         255843         100.00
no_complemento               207694          81.18
no_email                      84786          33.14
nu_telefone                   29334          11.47
nome_estabelecimento              2           0.00
comp                              0           0.00
regiao                            0           0.00
razao_social                      0           0.00
cnes                              0           0.00
municipio                         0           0.00
co_tipo_unidade                   0           0.00
ds_tipo_unidade                   0           0.00
desc_natureza_juridica            0           0.00
natureza_juridica                 0           0.00
no_logradouro                     0           0.00
nu_endereco                       0           0.00
tp_gestao                         0           0.00
uf                                0           0.00
co_cep                         

In [15]:
#Duplicados
df_duplicados = df_raw[df_raw.duplicated(keep=False)]
print(f"Total de linhas duplicadas: {len(df_duplicados)}")

Total de linhas duplicadas: 0


6.2 Ver quantidade de competências:

In [16]:
print(df_raw.nunique().sort_values(ascending=True))

motivo_desabilitacao         0
tp_gestao                    3
desc_natureza_juridica       3
regiao                       5
co_tipo_unidade              5
ds_tipo_unidade              5
uti_queimado_sus             8
uti_queimado_exist          15
uti_coronariana_sus         19
uf                          27
uti_coronariana_exist       30
natureza_juridica           32
comp                        36
uti_neonatal_sus            39
uti_pediatrico_sus          40
uti_pediatrico_exist        51
uti_neonatal_exist          56
uti_adulto_sus              84
uti_total_sus              116
uti_adulto_exist           122
uti_total_exist            155
leitos_sus                 557
leitos_existentes          625
no_complemento             994
nu_endereco               1938
no_bairro                 2476
municipio                 3473
no_email                  6063
razao_social              6731
no_logradouro             7047
co_cep                    7198
cnes                      7686
nu_telef

In [17]:
def valores_unicos(df, coluna):
    valores_unicos = df[coluna].unique()
    print(f"Valores únicos na coluna '{coluna}':")
    print(valores_unicos)
    print("-" * 50)

In [18]:
valores_unicos(df_raw, "tp_gestao")
valores_unicos(df_raw, "desc_natureza_juridica")
valores_unicos(df_raw, "regiao")
valores_unicos(df_raw, "co_tipo_unidade")
valores_unicos(df_raw, "ds_tipo_unidade")
valores_unicos(df_raw, "uti_queimado_sus")

Valores únicos na coluna 'tp_gestao':
<StringArray>
['M', 'E', 'D']
Length: 3, dtype: str
--------------------------------------------------
Valores únicos na coluna 'desc_natureza_juridica':
<StringArray>
['HOSPITAL_PRIVADO', 'HOSPITAL_Pï¿½BLICO', 'HOSPITAL_FILANTRï¿½PICO']
Length: 3, dtype: str
--------------------------------------------------
Valores únicos na coluna 'regiao':
<StringArray>
['NORDESTE', 'NORTE', 'SUDESTE', 'CENTRO-OESTE', 'SUL']
Length: 5, dtype: str
--------------------------------------------------
Valores únicos na coluna 'co_tipo_unidade':
[ 5  7 20 15 21]
--------------------------------------------------
Valores únicos na coluna 'ds_tipo_unidade':
<StringArray>
[              'HOSPITAL GERAL',       'HOSPITAL ESPECIALIZADO',
         'PRONTO SOCORRO GERAL',                'UNIDADE MISTA',
 'PRONTO SOCORRO ESPECIALIZADO']
Length: 5, dtype: str
--------------------------------------------------
Valores únicos na coluna 'uti_queimado_sus':
[0 2 4 6 5 1 3 7]
----

6.3 Ver quantidade de competências existentes

In [19]:
df_raw["comp"].drop_duplicates().sort_values().tolist()

[202301,
 202302,
 202303,
 202304,
 202305,
 202306,
 202307,
 202308,
 202309,
 202310,
 202311,
 202312,
 202401,
 202402,
 202403,
 202404,
 202405,
 202406,
 202407,
 202408,
 202409,
 202410,
 202411,
 202412,
 202501,
 202502,
 202503,
 202504,
 202505,
 202506,
 202507,
 202508,
 202509,
 202510,
 202511,
 202512]

6.4 Ver se existe duplicidade no grão esperado:

In [20]:
df_raw.duplicated(["comp", "cnes"]).sum()

np.int64(0)

O grão esperado da tabela fato é uma linha por estabelecimento de saúde, identificado pelo CNES, em uma competência mensal, identificada pela coluna COMP.                                                          Se duplicated(["comp", "cnes"]).sum() der 0, isso é ótimo, porque mostra que COMP + CNES identifica cada linha.

7. Tranformação dos dados

7.1 Definir as colunas

In [25]:
COLUNAS_TEXTO = [
    "regiao",
    "uf",
    "municipio",
    "nome_estabelecimento",
    "razao_social",
    "tp_gestao",
    "ds_tipo_unidade",
    "desc_natureza_juridica",
    "no_logradouro",
    "nu_endereco",
    "no_complemento",
    "no_bairro",
    "nu_telefone",
    "no_email"
]

COLUNAS_CODIGO = {
    "comp",
    "cnes",
    "co_tipo_unidade",
    "natureza_juridica",
    "co_cep"
}

METRICAS_LEITOS = [
    "leitos_existentes",
    "leitos_sus",
    "uti_total_exist",
    "uti_total_sus",
    "uti_adulto_exist",
    "uti_adulto_sus",
    "uti_pediatrico_exist",
    "uti_pediatrico_sus",
    "uti_neonatal_exist",
    "uti_neonatal_sus",
    "uti_queimado_exist",
    "uti_queimado_sus",
    "uti_coronariana_exist",
    "uti_coronariana_sus"
]

COLUNAS_TEXTO → colunas descritivas                                     
COLUNAS_CODIGO_NORMALIZAR → códigos que precisam ser preservados como texto                       
METRICAS_LEITOS → medidas numéricas que serão somadas nas análises

7.2 Função principal da preparação

In [33]:
def preparar_leitos(df_raw):
    """
    Prepara a base bruta de leitos para modelagem dimensional.

    Essa função faz:
    1. Padronização de textos
    2. Preservação de códigos como texto
    3. Tratamento de nulos
    4. Correção de zeros à esquerda
    5. Conversão de métricas para inteiro
    6. Criação de colunas temporais a partir de COMP
    7. Criação da descrição do tipo de gestão
    """

    # Criamos uma cópia para não alterar diretamente o DataFrame original.
    df = df_raw.copy()

    #Tirando colunas com muitos nulos que não agragam valor
    df.drop("motivo_desabilitacao", axis=1, inplace=True)
    df.drop("no_complemento", axis=1, inplace=True)

    # Padronização das colunas de texto.
    # Aqui removemos espaços extras, transformamos tudo em minusculo
    # e substituímos valores nulos por "NAO_INFORMADO".
    for col in COLUNAS_TEXTO:
        if col in df.columns:
            df[col] = df[col].fillna("nao_informado")
            df[col] = (
                df[col]
                .astype("str")
                .str.strip()
                .str.lower()
            )

    # Padronização dos códigos.
    # Esses campos são tratados como texto, não como número.
    # Isso preserva zeros à esquerda e evita interpretações erradas.
    for col in COLUNAS_CODIGO:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype("str")
                .str.strip()
                .str.replace(r"\.0$", "", regex=True)
            )

    # Garante tamanho padrão dos códigos.
    # zfill completa com zeros à esquerda.
    # Exemplo: "123" vira "0000123" se o tamanho esperado for 7.
    df["comp"] = df["comp"].str.zfill(6)
    df["cnes"] = df["cnes"].str.zfill(7)
    df["co_tipo_unidade"] = df["co_tipo_unidade"].str.zfill(2)
    df["natureza_juridica"] = df["natureza_juridica"].str.zfill(4)
    df["co_cep"] = df["co_cep"].str.zfill(8)

    # Converte as métricas de leitos e UTIs para número inteiro.
    # Se algum valor vier inválido, ele vira nulo com errors="coerce".
    for col in METRICAS_LEITOS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(int)

    #Renomeando as colunas para melhor entendimento e padronização
    novos_nomes = {
    "comp": "competencia",
    "ds_tipo_unidade": "desc_tipo_unidade",
    "no_logradouro": "logradouro",
    "TP_GESTAO": "tipo_gestao",
    "nu_endereco":"num_endereco",
    "no_bairro":"bairro",
    "nu_telefone":"telefone",
    "no_email":"email",
    "co_tipo_unidade":"cod_tipo_unidade",
    "co_cep":"cep",
    }
    df = df.rename(columns=novos_nomes)

    return df

In [35]:
df_preparado = preparar_leitos(df_raw)

print(f"DataFrame preparado: {df_preparado.shape[0]} linhas e {df_preparado.shape[1]} colunas")
df_preparado.head()

DataFrame preparado: 255843 linhas e 32 colunas


,competencia,regiao,uf,municipio,cnes,nome_estabelecimento,razao_social,tp_gestao,cod_tipo_unidade,desc_tipo_unidade,...,uti_adulto_exist,uti_adulto_sus,uti_pediatrico_exist,uti_pediatrico_sus,uti_neonatal_exist,uti_neonatal_sus,uti_queimado_exist,uti_queimado_sus,uti_coronariana_exist,uti_coronariana_sus
0,202301,nordeste,pe,cabo de santo agostinho,0000027,casa de saude santa helena,casa de saude e maternidade santa helena ltda,m,05,hospital geral,...,0,0,0,0,0,0,0,0,0,0
1,202301,nordeste,pe,cabo de santo agostinho,0000035,hospital mendo sampaio,prefeitura municipal do cabo de santo agostinho,m,05,hospital geral,...,0,0,0,0,0,0,0,0,0,0
2,202301,nordeste,pe,cabo de santo agostinho,0000094,maternidade padre geraldo leite bastos,prefeitura municipal do cabo de santo agostinho,m,07,hospital especializado,...,0,0,0,0,0,0,0,0,0,0
3,202301,nordeste,pe,cabo de santo agostinho,0000183,hospital samaritano,sociedade hospitalar samaritano ltda,m,05,hospital geral,...,5,0,0,0,0,0,0,0,0,0
4,202301,nordeste,pe,cabo de santo agostinho,0000221,hospital sao sebastiao,casa de saude e maternidade sao sebastiao ltda,m,05,hospital geral,...,10,0,0,0,0,0,0,0,0,0


7.3 Teste


In [ ]:
df_preparado[["tp_gestao", "descricao_gestao"]].drop_duplicates().sort_values("tp_gestao")

8. Validações de qualidade

8.1. Verificar duplicidade do grão

In [ ]:
duplicadas = df_preparado.duplicated(["competencia", "cnes"]).sum()

print("Duplicidades por competencia + cnes:", duplicadas)

Se der 0, significa que cada estabelecimento aparece uma única vez por mês.
Isso confirma o grão da fato.

8.2 Verificar se leitos SUS não ultrapassam leitos existentes


In [ ]:
erro_leitos_sus = (
    df_preparado["leitos_sus"] > df_preparado["leitos_existentes"]
).sum()

print("Linhas com leitos_sus > leitos_existentes:", erro_leitos_sus)

LEITOS_SUS não deveria ser maior que LEITOS_EXISTENTES.
Essa regra valida a consistência da base.

8.3 Verificar se UTI SUS não ultrapassa UTI existente

In [ ]:
erro_uti_sus = (
    df_preparado["uti_total_sus"] > df_preparado["uti_total_exist"]
).sum()

print("Linhas com uti_total_sus > uti_total_exist:", erro_uti_sus)

8.4. Verificar se UTI total bate com a soma dos tipos de UTI

In [ ]:
soma_uti_existente = (
    df_preparado["uti_adulto_exist"]
    + df_preparado["uti_pediatrico_exist"]
    + df_preparado["uti_neonatal_exist"]
    + df_preparado["uti_queimado_exist"]
    + df_preparado["uti_coronariana_exist"]
)

erro_soma_uti_existente = (
    df_preparado["uti_total_exist"] != soma_uti_existente
).sum()

print("Linhas com divergência na soma das UTIs existentes:", erro_soma_uti_existente)

8.5 Verificar se UTI sus total bate com a soma dos tipos de UTI sus

In [ ]:
soma_uti_sus = (
    df_preparado["uti_adulto_sus"]
    + df_preparado["uti_pediatrico_sus"]
    + df_preparado["uti_neonatal_sus"]
    + df_preparado["uti_queimado_sus"]
    + df_preparado["uti_coronariana_sus"]
)

erro_soma_uti_sus = (
    df_preparado["uti_total_sus"] != soma_uti_sus
).sum()

print("Linhas com divergência na soma das UTIs SUS:", erro_soma_uti_sus)

Além da limpeza dos dados, aplicamos regras de validação para garantir que as métricas fossem coerentes, como verificar se os leitos SUS não ultrapassam os leitos existentes e se o total de UTIs corresponde à soma das categorias específicas.

9. Criação das dimensões

9.1 Função auxiliar:

In [ ]:
def criar_dimensao(df, colunas, nome_id):
    """
    Cria uma dimensão a partir das combinações únicas das colunas escolhidas.

    Exemplo:
    Se a dimensão for localidade, usamos:
    regiao, uf, municipio

    Cada combinação única vira uma linha da dimensão.
    """

    dim = (
        df[colunas]
        .drop_duplicates()
        .sort_values(colunas, na_position="last")
        .reset_index(drop=True)
    )

    dim.insert(0, nome_id, range(1, len(dim) + 1))

    return dim

drop_duplicates → remove repetições         
sort_values → organiza a tabela                 
reset_index → cria índice limpo                     
insert → cria a chave substituta da dimensão                    

9.2 Dimensão tempo

In [ ]:
# Dicionário para transformar número do mês em nome do mês.
mapa_meses = {
    1: "janeiro",
    2: "fevereiro",
    3: "marco",
    4: "abril",
    5: "maio",
    6: "junho",
    7: "julho",
    8: "agosto",
    9: "setembro",
    10: "outubro",
    11: "novembro",
    12: "dezembro"
}

#Criando as colunas da dimensao tempo a partir da coluna COMP
df_preparado["data_competencia"] = pd.to_datetime(
    df_preparado["competencia"] .astype(str) + "01", format="%Y%m%d", errors="coerce")
df_preparado["ano"] = df_preparado["data_competencia"].dt.year.astype("Int64")
df_preparado["mes"] = df_preparado["data_competencia"].dt.month.astype("Int64")
df_preparado["trimestre"] = df_preparado["data_competencia"].dt.quarter.astype("Int64")
df_preparado["nome_mes"] = df_preparado["mes"].map(mapa_meses)



dim_tempo = criar_dimensao(
    df_preparado, 
    [
        "competencia", 
        "data_competencia", 
        "ano", 
        "mes", 
        "nome_mes", 
        "trimestre"
    ], 
    "id_tempo"
    )

dim_tempo.head()


Essa dimensão responde perguntas como:          
Quantos leitos havia em janeiro de 2025?            
Como os leitos evoluíram mês a mês?         
Qual foi o total por ano?       

9.3. Dimensão estabelecimento de saúde

In [ ]:
dim_estabelecimento_saude = criar_dimensao(
    df_preparado,
    [
        "cnes",
        "nome_estabelecimento",
        "razao_social",
        "logradouro",
        "endereco",
        "bairro",
        "cep",
        "telefone",
        "email"
    ],
    "id_estabelecimento"
)

dim_estabelecimento_saude.head()

CNES identifica o estabelecimento.                                                  
Nome, razão social e endereço descrevem esse estabelecimento.

9.4 Dimensão localidade

In [ ]:
dim_localidade = criar_dimensao(
    df_preparado,
    [
        "regiao",
        "uf",
        "municipio"
    ],
    "id_localidade"
)

dim_localidade.head()

Essa dimensão permite consultas por:                
região          
estado          
município                                                             

9.5. Dimensão tipo de unidade

In [ ]:
dim_tipo_unidade = criar_dimensao(
    df_preparado,
    [
        "cod_tipo_unidade",
        "desc_tipo_unidade"
    ],
    "id_tipo_unidade"
)

dim_tipo_unidade.head()

Exemplos de tipo de unidade:

HOSPITAL GERAL,                 
HOSPITAL ESPECIALIZADO,                 
PRONTO SOCORRO GERAL,               
UNIDADE MISTA               

9.6. Dimensão jurídica

In [ ]:
dim_natureza_juridica = criar_dimensao(
    df_preparado,
    [
        "natureza_juridica",
        "desc_natureza_juridica"
    ],
    "id_natureza_juridica"
)

dim_natureza_juridica.head()

Essa dimensão ajuda a comparar:

hospital público    
hospital privado    
hospital filantrópico   
outras categorias       

9.7. Dimensão gestão

In [ ]:
   
# Dicionário de domínio para o tipo de gestão.
# Isso deixa as consultas mais claras.
# Em vez de mostrar apenas M, E ou D, mostramos a descrição.
mapa_gestao = {
    "m": "municipal",
    "e": "estadual",
    "d": "dupla"
}

# Cria a descrição textual da gestão.
# Valores que não estiverem no dicionário viram NAO_INFORMADO.
df_preparado["desc_gestao"] = (
    df_preparado["tipo_gestao"]
    .fillna("nao informado")  
    .map(mapa_gestao)     
)


dim_gestao = criar_dimensao(
    df_preparado,
    [
        "tipo_gestao", 
        "desc_gestao"
    ],
    "id_gestao"
)

dim_gestao.head()

10. Criação da tabela fato

A fato será:                

fato_leitos_mensais         

O grão dela é:          

uma linha por estabelecimento de saúde em uma competência mensal                

In [ ]:
fato = df_preparado.copy()

fato = fato.merge(
    dim_tempo[["id_tempo", "comp"]],
    on="comp",
    how="left"
)

fato = fato.merge(
    dim_estabelecimento_saude[
        [
            "id_estabelecimento",
            "cnes",
            "nome_estabelecimento",
            "razao_social",
            "logradouro",
            "endereco",
            "bairro",
            "cep",
            "telefone",
            "email"
        ]
    ],
    on=[
        "cnes",
        "nome_estabelecimento",
        "razao_social",
        "logradouro",
        "endereco",
        "bairro",
        "cep",
        "telefone",
        "email"
    ],
    how="left"
)

fato = fato.merge(
    dim_localidade[
        [
            "id_localidade",
            "regiao",
            "uf",
            "municipio"
        ]
    ],
    on=["regiao", "uf", "municipio"],
    how="left"
)

fato = fato.merge(
    dim_tipo_unidade[
        [
            "id_tipo_unidade",
            "cod_tipo_unidade",
            "desc_tipo_unidade"
        ]
    ],
    on=["cod_tipo_unidade", "desc_tipo_unidade"],
    how="left"
)

fato = fato.merge(
    dim_natureza_juridica[
        [
            "id_natureza_juridica",
            "natureza_juridica",
            "desc_natureza_juridica"
        ]
    ],
    on=["natureza_juridica", "desc_natureza_juridica"],
    how="left"
)

fato = fato.merge(
    dim_gestao[["id_gestao", "tipo_gestao", "desc_gestao"]],
    on=["tipo_gestao", "desc_gestao"],
    how="left"
)

Explicação:             
                        
Cada merge procura o registro correspondente na dimensão.               
Depois disso, a fato recebe os IDs:             
id_tempo                        
id_estabelecimento                      
id_localidade                               
id_tipo_unidade             
id_natureza_juridica                    
id_gestao                               

10.1  Selecionar as colunas finais da fato

In [ ]:
colunas_fato = [
    "id_tempo",
    "id_estabelecimento",
    "id_localidade",
    "id_tipo_unidade",
    "id_natureza_juridica",
    "id_gestao"
] + METRICAS_LEITOS

fato_leitos_mensais = fato[colunas_fato]

fato_leitos_mensais.head()

10.2 Verificar se algum id ficou nulo

In [ ]:
fato_leitos_mensais[
    [
        "id_tempo",
        "id_estabelecimento",
        "id_localidade",
        "id_tipo_unidade",
        "id_natureza_juridica",
        "id_gestao"
    ]
].isna().sum()

11. Carregar no PostgreSQL

In [ ]:
load_dotenv()

user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
db = os.getenv('DB_NAME')
port = os.getenv('DB_PORT')
host = os.getenv('DB_HOST')

DATABASE_URL = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}"

engine = create_engine(DATABASE_URL)

11.1 Criar schemas:

In [ ]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS etl;"))

11.2 Enviar tabelas:

In [ ]:
dim_tempo.to_sql(
    "dim_tempo",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)

dim_estabelecimento_saude.to_sql(
    "dim_estabelecimento_saude",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)

dim_localidade.to_sql(
    "dim_localidade",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)

dim_tipo_unidade.to_sql(
    "dim_tipo_unidade",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)

dim_natureza_juridica.to_sql(
    "dim_natureza_juridica",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)

dim_gestao.to_sql(
    "dim_gestao",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)

fato_leitos_mensais.to_sql(
    "fato_leitos_mensais",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)